In [ ]:
import sys
import subprocess

def ensure_installed(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

ensure_installed(['transformers', 'datasets', 'torch', 'scikit-learn', 'pandas', 'numpy'])


In [ ]:
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import pipeline, AutoConfig
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
    pipeline_device = torch.device('mps')
else:
    device = 'cpu'
    pipeline_device = -1

print({'selected_device': device})


In [ ]:
dataset = load_dataset('dair-ai/emotion', split='validation')
class_names = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

subset_size = 240
seed = 42

dataset_df = dataset.to_pandas()
subset_df = (
    dataset_df.groupby('label', group_keys=False)
    .apply(lambda g: g.sample(n=max(1, subset_size // len(class_names)), random_state=seed))
    .reset_index(drop=True)
)

remaining = subset_size - len(subset_df)
if remaining > 0:
    used_texts = set(subset_df['text'].tolist())
    extra_df = dataset_df.loc[~dataset_df['text'].isin(used_texts)].sample(n=remaining, random_state=seed)
    subset_df = pd.concat([subset_df, extra_df], ignore_index=True)

subset_df = subset_df.sample(frac=1.0, random_state=seed).reset_index(drop=True)

texts = subset_df['text'].tolist()
true_ids = subset_df['label'].astype(int).tolist()

print({'split': 'validation', 'full_num_rows': len(dataset), 'subset_num_rows': len(subset_df), 'seed': seed})
print(subset_df.head(3).to_dict(orient='records'))
print({'subset_class_counts': subset_df['label'].value_counts().sort_index().to_dict()})


In [ ]:
model_name = 'bhadresh-savani/distilbert-base-uncased-emotion'
config = AutoConfig.from_pretrained(model_name)

clf = pipeline(
    task='text-classification',
    model=model_name,
    tokenizer=model_name,
    device=pipeline_device,
    truncation=True
)

id2label = {}
if hasattr(config, 'id2label') and config.id2label is not None:
    for k, v in config.id2label.items():
        id2label[int(k)] = str(v)

label2id = {}
if hasattr(config, 'label2id') and config.label2id is not None:
    label2id = {str(k): int(v) for k, v in config.label2id.items()}

print({'model_name': model_name, 'id2label': id2label, 'label2id': label2id})


In [ ]:
canonical_set = set(class_names)
alias_map = {
    'sadness': 'sadness',
    'sad': 'sadness',
    'joy': 'joy',
    'happy': 'joy',
    'love': 'love',
    'anger': 'anger',
    'angry': 'anger',
    'fear': 'fear',
    'scared': 'fear',
    'surprise': 'surprise',
    'surprised': 'surprise'
}

def normalize_label(raw_label):
    label = str(raw_label).strip()
    upper_label = label.upper()
    if upper_label.startswith('LABEL_'):
        idx = int(label.split('_')[-1])
        label = id2label.get(idx, label)
    label = str(label).strip().lower()
    label = alias_map.get(label, label)
    if label not in canonical_set:
        raise ValueError(f'Unrecognized label: {raw_label} -> {label}')
    return label

label_to_id = {name: i for i, name in enumerate(class_names)}
print({'class_names': class_names, 'label_to_id': label_to_id})


In [ ]:
batch_size = 32
pred_outputs = clf(texts, batch_size=batch_size, top_k=1)

pred_labels = []
pred_scores = []
for item in pred_outputs:
    if isinstance(item, list):
        item = item[0]
    pred_labels.append(normalize_label(item['label']))
    pred_scores.append(float(item['score']))

pred_ids = [label_to_id[label] for label in pred_labels]

results_df = pd.DataFrame({
    'text': texts,
    'true_label': [class_names[i] for i in true_ids],
    'predicted_label': pred_labels,
    'score': pred_scores
})
results_df['correct'] = results_df['true_label'] == results_df['predicted_label']

print(results_df.head(10).to_dict(orient='records'))


In [ ]:
accuracy = accuracy_score(true_ids, pred_ids)
report_dict = classification_report(true_ids, pred_ids, target_names=class_names, digits=4, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()
report_text = classification_report(true_ids, pred_ids, target_names=class_names, digits=4)
cm = confusion_matrix(true_ids, pred_ids)
cm_df = pd.DataFrame(cm, index=[f'true_{c}' for c in class_names], columns=[f'pred_{c}' for c in class_names])

print({
    'model_name': model_name,
    'dataset': 'dair-ai/emotion',
    'split': 'validation',
    'subset_num_examples': len(subset_df),
    'device': device,
    'accuracy': round(float(accuracy), 6)
})
print(report_text)
print(cm_df.to_string())
print(report_df.round(4).to_string())


In [ ]:
errors_df = results_df.loc[~results_df['correct'], ['text', 'true_label', 'predicted_label', 'score']].copy()
errors_df['score_gap_from_1'] = 1.0 - errors_df['score']

representative_errors_df = (
    errors_df.groupby(['true_label', 'predicted_label'], group_keys=False)
    .apply(lambda g: g.sort_values(by='score', ascending=False).head(2))
    .sort_values(by=['true_label', 'predicted_label', 'score'], ascending=[True, True, False])
    .reset_index(drop=True)
)

if len(representative_errors_df) < 20:
    additional_needed = 20 - len(representative_errors_df)
    existing_keys = set(representative_errors_df['text'].tolist())
    extra_errors_df = errors_df.loc[~errors_df['text'].isin(existing_keys)].sort_values(by='score', ascending=False).head(additional_needed)
    representative_errors_df = pd.concat([representative_errors_df, extra_errors_df], ignore_index=True)

representative_errors_df = representative_errors_df.head(20).copy()
representative_errors_df['text'] = representative_errors_df['text'].map(lambda x: x if len(x) <= 140 else x[:137] + '...')

print({'num_errors': int((~results_df['correct']).sum()), 'showing_representative_errors': len(representative_errors_df)})
print(representative_errors_df.to_string(index=False))
